
# **Simple Object Tracking by Color**

### **Objectives**
1. How to use an HSV Color Filter to Create a Mask and then Track our Desired Object


> **Assets note.** The original asset pack for this course was never committed to this repo (no `images/`, `videos/`, or `haarcascades/`, in any past commit). This fork ships a real, working replacement set — sourced from OpenCV's own official sample data where a good match existed, generated procedurally where it didn't (see `scripts/generate_assets.py`) — under `assets/images/`, `assets/videos/`, and `assets/haarcascades/`, plus small, called-out fixes wherever a swap needed one. Nothing else was removed. See the README for the full source list. No download step needed — just run the cells top to bottom.

In [1]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

In [2]:
# Download video
# !wget -O car-detection.mp4 "https://drive.google.com/file/d/1BRn36jqK8gG7vqJwVU9jCG-iLUz1w_wX/view?usp=drive_link"

In [3]:
#Object Tracking
import cv2
import numpy as np

# Initalize camera
#cap = cv2.VideoCapture(0)

# define range of color in HSV
# car-detection.mp4 (see the README for source/license) is the same clip
# OpenCV's own meanshift/CAMSHIFT tutorials use — its tracked car is a
# light silver sedan, so we threshold for low saturation + high value
# (i.e. "pale/light-colored"), not a specific hue.
lower = np.array([0, 0, 200])     # Lower bound for hue, saturation, and value
upper = np.array([179, 50, 255])  # Upper bound for hue, saturation, and value

# Create empty points array
points = []

# Get default camera window size

# Load video stream, long clip
cap = cv2.VideoCapture('assets/videos/car-detection.mp4')

# Get the height and width of the frame (required to be an interger)
width = int(cap.get(3)) 
height = int(cap.get(4))

# Define the codec and create VideoWriter object. The output is stored in '*.avi' file.
out = cv2.VideoWriter('car-detection_output.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30, (width, height))

ret, frame = cap.read()
Height, Width = frame.shape[:2]
frame_count = 0
radius = 0

while True:
  
    # Capture webcame frame
    ret, frame = cap.read()
    if ret:
      hsv_img = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

      # Threshold the HSV image to get only green colors
      mask = cv2.inRange(hsv_img, lower, upper)
      #mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
      
      contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
      
      # Create empty centre array to store centroid center of mass
      center =   int(Height/2), int(Width/2)

      if len(contours) > 0:
          
          # Get the largest contour and its center 
          c = max(contours, key=cv2.contourArea)
          (x, y), radius = cv2.minEnclosingCircle(c)
          M = cv2.moments(c)
          
          # Sometimes small contours of a point will cause a divison by zero error
          try:
              center = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))

          except:
              center =   int(Height/2), int(Width/2)

          # Allow only countors that have a larger than 25 pixel radius
          if radius > 25:
              
              # Draw cirlce and leave the last center creating a trail
              cv2.circle(frame, (int(x), int(y)), int(radius),(0, 0, 255), 2)
              cv2.circle(frame, center, 5, (0, 255, 0), -1)
              
          # Log center points 
          points.append(center)
      
      # If radius large enough, we use 25 pixels
      if radius > 25:
          
          # loop over the set of tracked points
          for i in range(1, len(points)):
              try:
                  cv2.line(frame, points[i - 1], points[i], (0, 255, 0), 2)
              except:
                  pass
              
          # Make frame count zero
          frame_count = 0
              
      out.write(frame)
    else:
      break

# Release camera and close any open windows
cap.release()
out.release()

In [4]:
!ffmpeg -i car-detection_output.avi car-detection_output.mp4 -y

ffmpeg version 8.1.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.0.123.102)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.101 / 60. 26.101
  libavcodec     62. 28.101 / 62. 28.101
  libavformat    62. 12.101 / 62. 12.101
  libavdevice    62.  3.101 / 62.  3.101
  libavfilter    11. 14.101 / 11. 14.101
  libswscale      9.  5.101 /  9.  5.101
  libswresample   6.  3.101 /  6.  3.101
Input #0, avi, from 'car-detection_output.avi':
  Metadata:
    software        : Lavf60.3.100
  Duration: 00:00:30.43, start: 0.000000, bitrate: 6091 kb/s
  Stream #0:0: Video: mjpeg 

[out#0/mp4 @ 0xb8ac54180] video:2155KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.535927%
frame=  913 fps=0.0 q=-1.0 Lsize=    2166KiB time=00:00:30.36 bitrate= 584.3kbits/s speed=34.2x elapsed=0:00:00.88    
[libx264 @ 0xb8b030a80] frame I:4     Avg QP:19.34  size: 21584
[libx264 @ 0xb8b030a80] frame P:232   Avg QP:21.65  size:  5639
[libx264 @ 0xb8b030a80] frame B:677   Avg QP:26.38  size:  1198
[libx264 @ 0xb8b030a80] consecutive B-frames:  0.8%  0.9%  0.7% 97.7%
[libx264 @ 0xb8b030a80] mb I  I16..4:  8.7% 83.8%  7.5%
[libx264 @ 0xb8b030a80] mb P  I16..4:  3.0% 15.9%  0.6%  P16..4: 26.8% 16.4% 12.6%  0.0%  0.0%    skip:24.7%
[libx264 @ 0xb8b030a80] mb B  I16..4:  0.5%  1.5%  0.1%  B16..8: 36.6%  5.4%  1.1%  direct: 2.0%  skip:52.9%  L0:47.6% L1:48.0% BI: 4.5%
[libx264 @ 0xb8b030a80] 8x8 transform intra:79.4% inter:73.6%
[libx264 @ 0xb8b030a80] coded y,uvDC,uvAC intra: 44.3% 66.4% 8.4% inter: 9.0% 12.8% 1.8%
[libx264 @ 0xb8b030a80] i16 v,h,dc,

In [5]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open('car-detection_output.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

In [6]:
HTML("""
<video controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

### Worth a second look

- HSV hue wraps at 0/180 — red sits right at the seam. If you swap in your own red-ish object and tracking gets flaky, OR a second band near hue 170–180 together with this one (the same trick the color-filtration notebook used for red).
- Log `radius` per frame into a list and plot it — a proxy for how close/far the tracked object is from the camera over time.